# Car Damage Detection — Colab version

Converted from Kaggle. Data now comes from Kaggle via `kagglehub` instead of the built-in `/kaggle/input` mount.

**Before running:** you need a Kaggle account + API token so `kagglehub` can authenticate.
- Get a token at https://www.kaggle.com/settings → "Create New Token" (downloads `kaggle.json`).
- Easiest in Colab: upload `kaggle.json` when prompted the first time you download a dataset (kagglehub will ask), **or** store `KAGGLE_USERNAME`/`KAGGLE_KEY` as Colab secrets (key icon in the left sidebar) and it'll pick them up automatically.

In [ ]:
# Install kagglehub (Colab doesn't have it preinstalled)
!pip install -q kagglehub


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("anujms/car-damage-detection")
print("Path to dataset files:", path)


In [ ]:
# Explore what actually got downloaded — the Kaggle-mounted layout
# (/kaggle/input/datasets/anujms/car-damage-detection/...) doesn't necessarily
# match the kagglehub cache layout, so walk the tree instead of guessing.
import os

for root, dirs, files in os.walk(path):
    print(root, f"({len(files)} files)" if files else "")


In [ ]:
# Locate the training/validation folders dynamically, wherever kagglehub put them,
# instead of hardcoding a path like the original /kaggle/input/... one.
import os

train_path = None
val_path = None

for root, dirs, files in os.walk(path):
    base = os.path.basename(root)
    if base == "training":
        train_path = root
    elif base == "validation":
        val_path = root

if train_path is None or val_path is None:
    raise FileNotFoundError(
        f"Could not find 'training'/'validation' folders under {path}. "
        "Print the tree above and adjust the folder names if the dataset structure differs."
    )

print("train_path:", train_path)
print("val_path:  ", val_path)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(rescale=1./255)

train_data = datagen.flow_from_directory(
    train_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    # shuffle=True
)

val_data = datagen.flow_from_directory(
    val_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    # shuffle=True
)


In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(2, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_data, validation_data=val_data, epochs=5)


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

base_model.trainable = False

model2 = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(2, activation='softmax')
])

model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model2.fit(train_data, validation_data=val_data, epochs=10)


In [ ]:
from tensorflow.keras.applications import ResNet50

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))

base_model.trainable = False

model3 = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(2, activation='softmax')
])

model3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model3.fit(train_data, validation_data=val_data, epochs=5)


In [ ]:
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras import layers, models

base_model = EfficientNetB4(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

model4 = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(2, activation='softmax')
])

# NOTE: fixed from the original notebook, which accidentally compiled/fit model3 again here
model4.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model4.fit(train_data, validation_data=val_data, epochs=50)


In [ ]:
from tensorflow.keras.applications import InceptionV3

base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

model5 = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(2, activation='softmax')
])

model5.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model5.fit(train_data, validation_data=val_data, epochs=5)


In [ ]:
from tensorflow.keras.applications import DenseNet121

base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

model6 = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(2, activation='softmax')
])

model6.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model6.fit(train_data, validation_data=val_data, epochs=10)
